<a href="https://colab.research.google.com/github/ngonzo95/BarrenLandAnalysis/blob/master/Manually_Rate_Property.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import requests
import pandas as pd
from google.colab import userdata
import datetime


In [ ]:
api_key = userdata.get('stand_rater')
print("API key retrieved.")

API key retrieved.


# Uploading a New Rate set
How you can try uploading the next experimental rate set. All experimental rates are saved unless explicitly overwritten so only run this on a change.

## Important Steps to push new rates
Make sure to update the file path to the excel file with your new path and update the table name so we can use the right one

In [ ]:
import requests

url = "https://standex-dev.up.railway.app/internal/rater/v9/create_rate_table"
file_path = "/content/updated_cov_b_limits.xlsx"
rate_table_name = "updated_cov_b_limits"
overwrite_value = "true"

# Construct the curl command as a single-line f-string to avoid syntax errors
# Ensure proper quoting for shell arguments
curl_command = f'curl -X POST "{url}" -H "x-api-key: {api_key}" -F "file=@{file_path}" -F "name={rate_table_name}" -F "overwrite={overwrite_value}"'

print("Executing curl command...")
# Execute the curl command
get_ipython().system(curl_command)
print("Curl command executed.")

Executing curl command...
curl: (26) Failed to open/read local data from file/application
Curl command executed.


# Setting up you rater
Make sure you are using your desired rate table key. Must run these cells before anything else will work

In [ ]:
#Important Choose Wisely
rate_table_key = "HO_CA_Rating_Algorithm_v92.pkl"


In [ ]:
# curl_url = "https://standex-dev.up.railway.app/internal/rater/v9/experimental_rate_debug"
# curl_url = "https://standex-dev.up.railway.app/internal/rater/v8/rate_debug" #To run on v8
# curl_url = "https://standex-dev.up.railway.app/internal/rater/v9/experimental_rate_debug"
curl_url = "https://standex-dev.up.railway.app/internal/rater/v9/rate_debug"
def call_rater(info, deductible):
    url = curl_url
    data = {
      'rate_table_key': rate_table_key,
      'policy_info': info,
      'deductibles': deductible
    }

    headers = {'Content-Type': 'application/json', 'x-api-key': userdata.get('stand_rater')}
    response = requests.post(url, json=data, headers=headers)
    # print(response.text)
    json = response.json()
    # print(json)
    if 'rate_table' in json.keys():
      del json['rate_table']
    return json


def verbose_call_rater(info, deductible):
    url = curl_url
    # print(info)
    data = {
      'rate_table_key': rate_table_key,
      'policy_info': info,
      'deductibles': deductible
    }
    # print(data)

    headers = {'Content-Type': 'application/json', 'x-api-key': userdata.get('stand_rater')}
    response = requests.post(url, json=data, headers=headers)
    try:
      json = response.json()
      return json
    except Exception as e:
      print(response.text)
      raise e

def pretty_print_rate_table(rate_table_output):
    # Define the desired order of perils
    peril_order = ['P1', 'P2', 'P2-A', 'P2-B', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10']

    # Define a desired order for factors
    desired_factor_order = [
        'Base Rate',
        'Territory',
        'Wildfire Score',
        'ARF',
        'Tier Factor',
        'Const Type and PC',
        'Occupancy Type - Dwelling',
        'Deductible',
        'Age of Home',
        'Years Since Renovation',
        'Fire Protective Devices',
        'Theft Protective Devices',
        'Coverage A Limit',
        'Coverage B Limit',
        'Coverage C Limit',
        'Coverage D Limit',
        'Seasonal Residence',
        'Secondary Residence',
        'Months Unoccupied',
        'Wood, Coal, or Pellet Stove',
        'Log Home',
        'Roof Covering',
        'Retrofitted Dwelling',
        'Loss History',
        'Loss Free Discount',
        'Home Buyer Discount',
        'Age of Insured',
        'Coverage E and F Limits',
        'DIC Endorsement',
        'Miles to Fire Department',
        'Roof Attached Structures',
        'Multi-Policy',
        'Presence of Fence',
        'Presence of Pool',
        'Limited Water',
        'Extended Dwelling',
        'WF Mitigation',
        'AgreedValue',
        'Firewise',
        "Defensible Space",
        'Window Glass Type',
        'Rented to Other Surcharge',
        'Course of Construction',
        'Initial Fire Impact Probability',
        'Resilient Home Discount'
    ]

    # Collect all unique factor names across all perils
    all_factors_set = set()
    for peril_data in rate_table_output.values():
        all_factors_set.update(peril_data.keys())

    # Create the final factor order, prioritizing desired order, then alphabetical for the rest
    final_factor_order = [factor for factor in desired_factor_order if factor in all_factors_set]
    remaining_factors = sorted([factor for factor in all_factors_set if factor not in desired_factor_order])
    final_factor_order.extend(remaining_factors)

    # Prepare data for DataFrame
    df_data = {}
    for factor in final_factor_order:
        df_data[factor] = []
        for peril in peril_order:
            # Get the factor value for the current peril, or None if not present
            value = rate_table_output.get(peril, {}).get(factor, None)
            df_data[factor].append(value)

    # Create DataFrame
    df = pd.DataFrame(df_data, index=peril_order).T
    df.index.name = 'Factor'

    # Calculate the product for each peril (column)
    product_row = {}
    for col in df.columns:
        # Ensure we only multiply numeric values, treating None/non-numeric as 1
        numeric_values = pd.to_numeric(df[col], errors='coerce').fillna(1.0)
        product_row[col] = round(np.prod(numeric_values))

    # Append the product row to the DataFrame
    df.loc['Total Premium Before Expenses'] = product_row

    return df

In [ ]:
example_call =   {
        "deductibles": {
          "others": 25000,
          "water": 25000,
          "wildfire": 0.1
        }, "policy_info": {
  "form": "Modified HO-5",
  "zip": 91750,
  "gre_wf_score": 3.64,
  "gre_arf": 1.5,
  "construction_type": "Frame",
  "protection_class": "3",
  "year_built": 1995,
  "coverage_a": 3661000,
  "coverage_b": 100000,
  "coverage_c": 1000000,
  "coverage_d": 800000,
  "coverage_e": 500000,
  "coverage_f": 25000,
  "roof_covering": "Asphalt Fiberglass Composite",
  "insured_dob": "1953-05-14",
  "purchase_year": 2026,
  "miles_to_fire_department": 2,
  "fire_alarm": "Central",
  "sprinkler": "No",
  "fire_ext": "Yes",
  "smoke": "Yes",
  "theft_alarm": "Central",
  "dead_bolt": "Yes",
  "presence_of_fence": "Yes - non-combustible fence",
  "count_of_solar_panels": 0,
  "months_unoccupied": 0,
  "claims": [],
  "extended_dwelling_coverage_limit": 1,
  "water_damage_limit": 1,
  "retrofitted": "No",
  "wf_category": "C",
  "is_pay_in_full": True,
  "fireshed_capacity": 0.3333,
  "replacement_cost": 4411000,
  "is_dic": "No",
  "is_agreed_value": False,
  "multi_policy_discount": "No",
  "stand_protect_discount": "No",
  "ensemble_wf_score": 0.6385968819599109,
  "defensible_space": 0,
  "uw_percent_adjustment": 0,
  "start_date": "2026-01-13",
  "is_renovated_to_studs": "No",
  "is_firewise_community": "No",
  "window_glass_type": "Unknown",
  "rental_type": "No",
  "renovation_magnitude": "No",
  "year_renovated": 2000,
  "presence_of_pool": "Yes",
  "secondary_residence": "No",
  "seasonal": "No"
        }
        }


# Get the full response first
# Extract deductibles
deductibles_data = example_call['deductibles']

# Create policy_info by copying example_call and removing 'deductibles'
policy_info_data = example_call['policy_info']

# Call the rater
call_rater(policy_info_data, deductibles_data)

{'fac_fee': 6915.0,
 'min_premium_hit': 0,
 'peril_premium': {'P1': 847.0,
  'P10': 156.0,
  'P2': 6818.0,
  'P2-A': 10138.0,
  'P2-B': 758.0,
  'P3': 904.0,
  'P4': 58.0,
  'P5': 106.0,
  'P6': 5320.0,
  'P7': 133.0,
  'P8': 340.0,
  'P9': 1222.0},
 'simplified_premium': 30343.0,
 'total_premium': 32307.91}

## If you want to compare rate tables use this:

In [ ]:
# Get the full response first
rater_response = verbose_call_rater(example_call['policy_info'], example_call['deductibles'])
# Then pass only the 'rate_table' part to pretty_print_rate_table
pretty_print_rate_table(rater_response.get('rate_table', {}))

,P1,P2,P2-A,P2-B,P3,P4,P5,P6,P7,P8,P9,P10
Factor,,,,,,,,,,,,
Base Rate,3070.000,12947.000,14532.000,1438.000,1293.000,1677.000,621.000,12829.000,1106.000,1732.000,1612.000,419.000
Territory,0.809,0.787,1.000,0.787,0.832,0.814,0.539,0.818,0.767,0.903,1.759,0.834
Wildfire Score,1.000,1.063,1.000,1.063,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
ARF,1.000,1.000,0.805,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
Tier Factor,1.031,1.000,1.000,1.000,1.038,1.013,1.018,1.019,1.005,1.002,1.022,1.000
Const Type and PC,1.100,1.420,1.000,1.420,1.060,0.982,1.083,0.943,0.918,1.083,0.932,1.000
Occupancy Type - Dwelling,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
Deductible,0.975,0.910,0.910,0.910,1.000,0.120,0.369,0.599,0.319,0.359,0.689,0.672
Age of Home,1.160,1.064,1.000,1.064,1.083,1.024,1.005,1.334,1.094,1.005,1.167,1.040


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Runing a Notional book  


## Where you can upload the notional book
Colab does not store file state. You will have to upload a file every time you reload the page. Make sure there is no missing data and it follows this rough format

In [ ]:
# Make sure to upload 'v_8_rate_book.csv' to the /content/ directory in your Colab environment.
notional_book_df = pd.read_csv('/content/Quote data for Collab Rater 20260310 v2.csv', dtype={'presence_of_pool': 'string', 'year_renovated': 'string', 'miles_to_fire_department': 'float64', 'replacement_cost': 'float64','count_of_solar_panels': 'string', 'insured_dob': 'string' })

# Replace '-' with NaN across the DataFrame before numeric conversion
#notional_book_df = notional_book_df.replace('-', np.nan)

num_cols = notional_book_df.select_dtypes(include='number').columns
notional_book_df[num_cols] = notional_book_df[num_cols].astype('float64')

# notional_book_df = notional_book_df.iloc[0:1251]
notional_book_df

,Retool Property ID,gre_id,Address,City,zip,County,State,gre_wf_score,gre_arf,construction_type,...,window_glass_type,window_glass_type.1,rented_to_others,course_of_construction,payment_schedule_name,is_agreed_value,replacement_cost,is_renovated_to_studs,uw_percent_adjustment,start_date
0,8ba45abe-3def-44ff-85ec-6f06d80e7405,100291502.0,3676 Happy Valley Rd,Lafayette,94549.0,Contra Costa,CA,2.88,9.417000,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,4773000.0,No,0.00,11/6/2019
1,1703c1d1-73ca-4cb5-8e66-ae0c30944be5,100929546.0,26460 Tierra Vista Ln,Salinas,93908.0,Monterey,CA,3.50,0.000000,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,3421000.0,No,0.62,4/7/2021
2,b2fe32a2-d924-483e-9d31-12323f459251,100723312.0,1894 Westridge Rd,Los Angeles,90049.0,Los Angeles,CA,3.54,4.499380,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,5074000.0,No,0.00,4/17/2021
3,60c03e9f-4ffb-4a46-88c1-c6dd5ea8e970,100455264.0,12258 Richwood Dr,Los Angeles,90049.0,Los Angeles,CA,3.52,11.260000,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,3382000.0,No,0.00,12/5/2023
4,9a77ff28-4594-4bbb-92ec-4230d17d2663,100327276.0,4267 Beresford Way,La CaÃ±ada Flintridge,91011.0,Los Angeles,CA,3.43,1.652000,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,7536000.0,No,0.00,12/28/2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4993,761c9ad5-a8ff-4a30-ba39-b00773c565d2,100889156.0,9 Arastradero Road,Portola Valley,94028.0,Santa Clara,CA,2.50,0.000000,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,6000000.0,No,-0.06,7/1/2026
4994,38aeca59-68e1-45ec-9276-4c74bbf25f56,100431804.0,650 N Almar Ave,Los Angeles,90272.0,Los Angeles,CA,3.54,0.996000,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,5083000.0,No,0.00,11/24/2026
4995,1e44fc9c-fde3-40c6-a1df-a4ecdb7d5e47,100379550.0,458 Rockport Cir,Folsom,95630.0,Sacramento,CA,1.23,0.580000,Frame,...,Tempered Glass,Tempered Glass,No,No,upfront,No,3392000.0,No,-0.20,12/31/2026
4996,35bae0f0-10da-492e-aa59-f1ed4e22e75d,100937620.0,782 Glenside Dr,Lafayette,94549.0,Contra Costa,CA,1.94,0.133728,Frame,...,Unknown,Double Pane Non Tempered,No,No,upfront,No,3194000.0,No,0.00,4/24/2027


In [ ]:
import traceback
perils = ['P1', 'P2', 'P2-A', 'P2-B', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10',]

if curl_url == "https://standex-dev.up.railway.app/internal/rater/v8/rate_debug":
  perils.remove('P2-B')

info_keys = notional_book_df.columns.tolist()
deductible_keys = ['others', 'wildfire', 'water']

def get_rate(property, raw_return=False):
    try:
      deductible = {'others': property['others'], 'wildfire': property['wildfire'], 'water': property['water'] }
      info = {
          'form': 'Modified HO-5',
          'zip': property['zip'],
          'gre_wf_score': np.trunc(property['gre_wf_score'] * 100)/100,
          'gre_arf': np.round(property['gre_arf'], 3),
          'construction_type': property['construction_type'],
          'protection_class': property['protection_class'],
          'year_built': property['year_built'],
          'coverage_a': property['coverage_a'],
          'coverage_b': property['coverage_b'],
          'coverage_c': property['coverage_c'],
          'coverage_d': property['coverage_d'],
          'coverage_e': int(property['coverage_e']),
          'coverage_f': int(property['coverage_f']),
          'roof_covering': property['roof_covering'],
          'insured_dob': property['insured_dob'],
          'purchase_year': 'Default' if property['purchase_year'] == 'Default' else int(property['purchase_year']),
          'miles_to_fire_department': property['miles_to_fire_department'],
          'fire_alarm': property['fire_alarm'],
          'sprinkler': property['sprinkler'],
          'fire_ext': property['fire_ext'],
          'smoke': property['smoke'],
          'theft_alarm': property['theft_alarm'],
          'dead_bolt': property['dead_bolt'],
          'presence_of_fence': property['presence_of_fence'],
          'presence_of_pool': property['presence_of_pool'],
          'count_of_solar_panels': 0 if property['count_of_solar_panels'] == "0" else property['count_of_solar_panels'],
          'seasonal': property['seasonal'],
          'secondary_residence': property['secondary_residence'],
          'months_unoccupied': int(property['months_unoccupied']),
          'water_damage_limit': property['water_damage_limit'],
          'extended_dwelling_coverage_limit': property['extended_dwelling_coverage_limit'],
          'retrofitted': 'No',
          'claims': [],
          'wf_category': property['wf_category'],
          'is_pay_in_full': True,
          'fireshed_capacity': float(property['fireshed_capacity']),
          'is_agreed_value': True if property['is_agreed_value'] == 'Yes' else False,
          'replacement_cost': property['replacement_cost'],
          'is_dic': property['is_dic'],
          'multi_policy_discount': property['multi_policy_discount'],
          'stand_protect_discount': property['stand_protect_discount'],
          'year_renovated': int(property['year_built']) if property['renovation_year'] == 'Default' else int(property['renovation_year']),
          'ensemble_wf_score': 'Default' if property['Wf_Score_Mean'] == 'Default' else float(property['Wf_Score_Mean']),
          'defensible_space': property['defensible_space'],
          'uw_percent_adjustment': 0,
          'is_renovated_to_studs': 'No',
          'is_firewise_community': property['firewise'],
          'window_glass_type': property['window_glass_type'],
          'rental_type': property['rented_to_others'],
          'renovation_magnitude': property['course_of_construction'],
          'start_date': datetime.datetime.now().strftime("%Y-%m-%d")
      }
    except KeyError as e:
      missing_key = e.args[0]
      raise KeyError(
          f"Expected a column named '{missing_key}' but could not find it in the input property. Check column names on input book"
      ) from None

    if 'initial_pdr' in property:
      info['initial_pdr'] = property['initial_pdr']
    if 'post_mitigation_pdr' in property:
      info['post_mitigation_pdr'] = property['post_mitigation_pdr']


    info_return = [property[key] for key in info_keys]
    deductible_return = [deductible[key] for key in deductible_keys]

    try:
        rate = verbose_call_rater(info, deductible)
        if raw_return:
            return rate
            # return verbose_call_rater(info, deductible)

        premium_less_expense = np.round(pd.DataFrame(rate['rate_table']).prod())

        return_data = [rate['peril_premium'][peril] for peril in perils] + [premium_less_expense[peril] for peril in perils]
        # return_data['min_premium_hit'] = rate['min_premium_hit']
        return return_data + [rate['total_premium']] + [rate['fac_fee'], rate['min_premium_hit']] + info_return + deductible_return
    except Exception as e:
        print(e)
        stack = traceback.format_exc()
        print(stack)
        raise e

## Calling one from a notional book


In [ ]:
get_rate(notional_book_df.iloc[13], True)
#notional_book_df.iloc[13].to_dict()
#notional_book_df.iloc[13]['coverage_a']


{'fac_fee': 21700.0,
 'min_premium_hit': 0,
 'peril_premium': {'P1': 1045.0,
  'P10': 433.0,
  'P2': 21641.0,
  'P2-A': 22211.0,
  'P2-B': 2404.0,
  'P3': 2065.0,
  'P4': 105.0,
  'P5': 198.0,
  'P6': 12978.0,
  'P7': 281.0,
  'P8': 542.0,
  'P9': 1363.0},
 'rate_table': {'P1': {'ARF': 1.0,
   'Age of Home': 0.946,
   'Age of Insured': 0.909,
   'AgreedValue': 1.0,
   'Base Rate': 3070.0,
   'Const Type and PC': 1.1,
   'Course of Construction': 1.0,
   'Coverage A Limit': 1.72,
   'Coverage B Limit': 0.983,
   'Coverage C Limit': 0.729,
   'Coverage D Limit': 0.945,
   'Coverage E and F Limits': 1.0,
   'DIC Endorsement': 1,
   'Deductible': 0.975,
   'Defensible Space': 1.0,
   'Extended Dwelling': 1.0,
   'Fire Protective Devices': 0.6,
   'Firewise': 1.0,
   'Home Buyer Discount': 0.98,
   'Initial Fire Impact Probability': 1.0,
   'Limited Water': 1.0,
   'Log Home': 1.0,
   'Loss Free Discount': 0.758,
   'Loss History': 1,
   'Miles to Fire Department': 1.0,
   'Months Unoccupie

In [ ]:
premiums = []
for index, property in notional_book_df.iterrows():
    if property['gre_wf_score'] is None:
        print('skipped missing')
        print(index)
        continue
    try:
        premiums.append( get_rate(property))
    except Exception as e:
        print(f'error at index {index} with id {property["gre_id"]}')
        # print(index)
        # print(property)
    if index % 50 == 0:
      print(f'Completed {index} of {len(notional_book_df)}')

Completed 0 of 4998
Out of range float values are not JSON compliant
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 510, in prepare_body
    body = complexjson.dumps(json, allow_nan=False)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/simplejson/__init__.py", line 378, in dumps
    return _default_encoder.encode(obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/simplejson/encoder.py", line 298, in encode
    chunks = self.iterencode(o)
             ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/simplejson/encoder.py", line 379, in iterencode
    return _iterencode(o, 0)
           ^^^^^^^^^^^^^^^^^
ValueError: Out of range float values are not JSON compliant

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_297/1268335306.py", line 8

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(premiums,columns=perils + [f'{peril} Less Expense' for peril in perils] + ['total_with_fees'] + ['fac_fee', 'min_premium_hit'] + info_keys + deductible_keys)
df = df.set_index('gre_id')
df = df.dropna(how='all').reset_index(drop=True)
df

,P1,P2,P2-A,P2-B,P3,P4,P5,P6,P7,P8,...,is_agreed_value,og_is_agreed_value,replacement_cost,is_renovated_to_studs,og_is_renovated_to_studs,uw_percent_adjustment,start_date,others,wildfire,water
0,927.0,2894.0,2490.0,322.0,1638.0,192.0,976.0,6754.0,383.0,2870.0,...,No,0,3589000.0,No,0,0.00,12/20/24,10000.0,0.25,25000.0
1,709.0,64.0,1332.0,7.0,1483.0,143.0,313.0,3898.0,159.0,617.0,...,No,0,3715000.0,No,0,0.00,1/3/25,25000.0,25000.00,25000.0
2,934.0,51714.0,41884.0,5743.0,1123.0,76.0,258.0,4133.0,122.0,1242.0,...,No,0,2732000.0,No,0,0.00,2/26/25,25000.0,0.10,50000.0
3,369.0,18341.0,56342.0,2037.0,422.0,114.0,57.0,1976.0,67.0,392.0,...,No,0,4150000.0,No,0,0.00,3/10/25,25000.0,0.10,25000.0
4,483.0,33.0,1251.0,4.0,719.0,19.0,56.0,2593.0,18.0,304.0,...,No,0,1710000.0,No,0,0.00,2/15/25,50000.0,50000.00,50000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,902.0,1154.0,716.0,128.0,1880.0,201.0,421.0,2547.0,293.0,472.0,...,No,No,2000000.0,No,No,-0.20,12/7/25,10000.0,10000.00,25000.0
637,491.0,1448.0,14063.0,161.0,1112.0,100.0,329.0,3212.0,170.0,225.0,...,No,No,4411000.0,Yes,Yes,-0.05,12/10/25,25000.0,25000.00,25000.0
638,284.0,1071.0,784.0,119.0,901.0,66.0,114.0,5405.0,163.0,394.0,...,No,No,2396000.0,Yes,Yes,-0.20,12/11/25,25000.0,25000.00,25000.0
639,921.0,7763.0,12453.0,862.0,1805.0,36.0,188.0,6859.0,34.0,667.0,...,No,No,7434000.0,No,No,0.00,12/12/25,50000.0,0.05,50000.0


In [ ]:
#df.to_csv('/content/cfd_asis_quote_output.csv', index=False)
df.to_csv('/content/notional_book_v9_rates.csv', index=False)

# Direct Comparing rater to spreadsheet
Sometimes it is helpful to be able to directly compare whats in the google sheet to what the rater is computing to make sure the intent aligns. this automaticalls pulls input and output from the HO CA sheet and compares factor by factor for you

In [ ]:
import gspread
from google.colab import auth
from gspread_dataframe import get_as_dataframe
import google.auth

# Authenticate with Google Colab
auth.authenticate_user()

# Authorize gspread to interact with Google Sheets using the default credentials
# and explicitly adding 'https://www.googleapis.com/auth/drive' scope for shared drive access
credentials, project_id = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
gc = gspread.Client(auth=credentials)

print("Authenticated and gspread client initialized.")

Authenticated and gspread client initialized.


## Inputs
This pulls the current active input from the google sheet and then transforms it into the desired payload so we can call the automated algo


In [ ]:
import datetime

def algo_rates():
    spreadsheet_name = 'HO CA Rating Algorithm and Rate Tables(Main)'
    input_worksheet_name = 'Input' # The worksheet to read from

    try:
        spreadsheet = gc.open(spreadsheet_name)
        input_worksheet = spreadsheet.worksheet(input_worksheet_name)

        data_range = input_worksheet.get('A:C')

        # Temporary dict to store all parsed policy info from the sheet before filtering
        raw_policy_info = {}
        deductibles = {}

        # Define mapping for sheet names to rater API names
        policy_info_mapping = {
            'Zip Code': 'zip',
            'Wildfire Category': 'wf_category',
            '% Capacity': 'fireshed_capacity',
            'GRe Wildfire Score': 'gre_wf_score',
            'Ensemble Score': 'ensemble_wf_score',
            'GRe ARF': 'gre_arf',
            'Initial PDR': 'initial_pdr',
            'Change from Initial PDR': 'post_mitigation_pdr',
            'Year home purchased': 'purchase_year',
            'Construction Type': 'construction_type',
            'Protection Class': 'protection_class',
            'Year Built': 'year_built',
            'Renovated To The Studs': 'is_renovated_to_studs',
            'Fire Protective Device - Alarm': 'fire_alarm',
            'Fire Protective Device - Sprinklers': 'sprinkler',
            'Fire Protective Device - Fire Extinguisher': 'fire_ext',
            'Fire Protective Device - Smoke Deductors': 'smoke',
            'Theft Protective Devices - Alarm': 'theft_alarm',
            'Theft Protective Devices - Deadbolt': 'dead_bolt',
            'Agreed Value (yes/no)': 'is_agreed_value',
            'Replacement Cost': 'replacement_cost',
            'Coverage A Limit (Dwelling)': 'coverage_a',
            'Coverage B Limit (Other Structures)': 'coverage_b',
            'Coverage C Limit (Contents)': 'coverage_c',
            'Coverage D Limit (Loss of Use)': 'coverage_d',
            'Seasonal Residence': 'seasonal',
            'Secondary Residence': 'secondary_residence',
            'Months Unoccupied (Secondary or Seasonal Dwelling)': 'months_unoccupied',
            'Roof Covering': 'roof_covering',
            'Retrofitted Dwelling Factor': 'retrofitted',
            'Age of Insured (input is birthdate?)': 'insured_dob',
            'Coverage E (Liability)': 'coverage_e',
            'Coverage F (medical)': 'coverage_f',
            'Difference in Conditions Endorsement': 'is_dic',
            'Miles to Fire Department': 'miles_to_fire_department',
            'Solar Panels': 'count_of_solar_panels',
            'Multi-Policy Discount (other Stand Policies?)': 'multi_policy_discount',
            'Presence of Fence': 'presence_of_fence',
            'Presence of Pool': 'presence_of_pool',
            'Limited Water Damage Coverage': 'water_damage_limit',
            'Extended Dwelling Coverage': 'extended_dwelling_coverage_limit',
            'Stand Protect Discount': 'stand_protect_discount',
            'Firewise Community': 'is_firewise_community',
            'Defensible space': 'defensible_space',
            'Window Glass Type': 'window_glass_type',
            'Rented to Others': 'rental_type',
            'Course of Construction': 'renovation_magnitude',
            'UW Special Adjustment': 'uw_percent_adjustment',
            # These are for specific rater inputs not directly tied to default names
            'Total Insured Value': 'tiv',
            'Initial PDR': 'initial_pdr',
            'Change from Initial PDR': 'post_mitigation_pdr',
            'Year Renovated': 'year_renovated' # Assuming there's a Year Renovated field
        }

        deductible_mapping = {
            'AOP Deductible': 'others',
            'Water Deductible': 'water',
            'WF Deductible': 'wildfire'
        }

        for row in data_range[1:]:
            if len(row) >= 3:
                key_sheet = row[0].strip()
                value_str = row[2].strip()

                if not key_sheet:
                    continue

                value = None
                # Try to parse special strings
                if value_str.lower() == 'true':
                    value = True
                elif value_str.lower() == 'false':
                    value = False
                elif value_str == 'None' or value_str == '':
                    value = None
                elif value_str.lower() == 'default':
                    value = 'Default' # Preserve 'Default' string
                else:
                    # Clean numeric strings by removing commas, then try converting
                    clean_value_str = value_str.replace(',', '')
                    try:
                        if '.' in clean_value_str:
                            value = float(clean_value_str)
                        else:
                            value = int(clean_value_str)
                    except ValueError:
                        # Handle percentage strings like '10%' for deductibles
                        if clean_value_str.endswith('%'):
                            try:
                                value = float(clean_value_str.replace('%', '')) / 100.0
                            except ValueError:
                                value = value_str # Keep as string if parsing fails
                        else:
                            value = value_str # Keep as string if no other parsing works

                # Apply mapping
                if key_sheet in deductible_mapping:
                    deductibles[deductible_mapping[key_sheet]] = value
                elif key_sheet in policy_info_mapping:
                    raw_policy_info[policy_info_mapping[key_sheet]] = value
                # Else, ignore fields not explicitly mapped to rater inputs

        # Get the keys from the example_call['policy_info'] to filter against
        # NOTE: 'example_call' must be defined in the notebook's global scope for this to work.
        # If example_call is not available, you would need a static list of expected keys.
        expected_rater_policy_keys = set(example_call['policy_info'].keys())

        # Filter raw_policy_info to only include keys present in the expected_rater_policy_keys
        policy_info = {k: v for k, v in raw_policy_info.items() if k in expected_rater_policy_keys}

        # Add default values for missing rater fields if necessary and not already set
        if 'claims' not in policy_info: policy_info['claims'] = []
        if 'is_pay_in_full' not in policy_info: policy_info['is_pay_in_full'] = True
        if 'form' not in policy_info: policy_info['form'] = 'Modified HO-5'
        if 'year_renovated' not in policy_info: policy_info['year_renovated'] = int(policy_info['year_built'])
        # Convert 'is_agreed_value' to boolean if it's 'Yes'/'No'
        if 'is_agreed_value' in policy_info and isinstance(policy_info['is_agreed_value'], str):
            policy_info['is_agreed_value'] = policy_info['is_agreed_value'].lower() == 'yes'
        # Set start_date if not explicitly provided or mapped (example_call uses a specific date format)
        if 'start_date' not in policy_info:
             policy_info['start_date'] = datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S.000-0700")

        if policy_info['count_of_solar_panels'] == 'Yes':
            policy_info['count_of_solar_panels'] = "1+"
        else:
            policy_info['count_of_solar_panels'] = 0

        # Reconstruct the expected dictionary structure for the rater
        rater_input_dict = {
            'deductibles': deductibles,
            'policy_info': policy_info
        }

        return rater_input_dict

    except gspread.exceptions.WorksheetNotFound:
        print(f"Error: Worksheet '{input_worksheet_name}' not found in spreadsheet '{spreadsheet_name}'.")
        return None
    except gspread.exceptions.SpreadsheetNotFound:
        print(f"Error: Spreadsheet '{spreadsheet_name}' not found. Please check the name.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [ ]:
sheets_input_payload = algo_rates()

# Get the full response first
rater_response_from_sheets = verbose_call_rater(sheets_input_payload['policy_info'], sheets_input_payload['deductibles'])

# Then pass only the 'rate_table' part to pretty_print_rate_table,
# providing an empty dict if 'rate_table' is missing to prevent KeyError.
automated_rt = pretty_print_rate_table(rater_response_from_sheets.get('rate_table', {}))
display(automated_rt)

KeyError: 'rate_table'

## This pulls the rate table from the google sheet directly so that you can compare its output

In [ ]:
import gspread
from google.colab import auth
from gspread_dataframe import get_as_dataframe
import google.auth
import re # Import regex module

# Authenticate with Google Colab
auth.authenticate_user()

# Authorize gspread to interact with Google Sheets using the default credentials
# and explicitly adding 'https://www.googleapis.com/auth/drive' scope for shared drive access
credentials, project_id = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
gc = gspread.Client(auth=credentials)

spreadsheet_name = 'HO CA Rating Algorithm and Rate Tables(Main)'

try:
    # Open the spreadsheet by its title
    spreadsheet = gc.open(spreadsheet_name)

    # Select the worksheet
    worksheet = spreadsheet.worksheet('Homeowners Rating')

    # Get values from the specified range B10:P57
    data = worksheet.get('B10:P57')

    # The first row of the range (data[0]) contains headers like ['Description', 'Form Number', '', 'P1', ...]
    headers = data[0]

    # Extract actual peril column names (e.g., 'P1', 'P2', ...) starting from index 3 in the headers list.
    # We skip 'Description', 'Form Number', and the empty column between 'Form Number' and 'P1'.
    peril_columns_names = [h.strip() for h in headers[3:] if h.strip()]

    # Prepare data for DataFrame
    processed_rows_data = []
    df_index_list = []

    for row in data[1:]: # Iterate over data rows (from B11:P57)
        # Extract factor name from column B (index 0 in the raw row) and clean it
        factor_name_raw = row[0].strip()

        # Clean factor name: remove any combination of spaces, digits, or non-word characters from the end.
        # Example: 'Territory Factor        1205' -> 'Territory Factor'
        cleaned_factor_name = re.sub(r'[\s\d\W]+$', '', factor_name_raw).strip()

        # Fallback if the cleaning makes the name empty (e.g., if it was just numbers)
        if not cleaned_factor_name:
            cleaned_factor_name = factor_name_raw # Use original if cleaning resulted in empty string

        df_index_list.append(cleaned_factor_name)

        # Extract peril values, starting from index 3 in the raw row (corresponding to 'P1' onwards)
        peril_values_raw = row[3:]

        # Clean and convert values to numeric
        cleaned_peril_values = []
        for val in peril_values_raw:
            try:
                # Remove commas and convert to float
                cleaned_peril_values.append(float(str(val).replace(',', '')))
            except ValueError:
                # If conversion fails (e.g., empty string, other text), append NaN
                cleaned_peril_values.append(np.nan)

        processed_rows_data.append(cleaned_peril_values)

    # Create DataFrame with cleaned index and columns
    sheets_df = pd.DataFrame(processed_rows_data, index=df_index_list, columns=peril_columns_names)
    sheets_df.index.name = 'Factor' # Name the index column 'Factor'

    # Define mapping from spreadsheet factor names to desired names for consistency
    # This mapping should be exhaustive for all factors you want to match
    factor_name_mapping = {
        'Territory Factor': 'Territory',
        'Area Rating Factor': 'ARF',
        'Construction Type and Protection Class': 'Const Type and PC',
        'Occupancy Type': 'Occupancy Type - Dwelling',
        'Deductible Factor': 'Deductible',
        'Age of Home': 'Age of Home',
        'Years Since Renovation': 'Years Since Renovation',
        'Fire Protective Devices': 'Fire Protective Devices',
        'Theft Protective Devices': 'Theft Protective Devices',
        'Coverage A Limit': 'Coverage A Limit',
        'Coverage B Limit': 'Coverage B Limit',
        'Coverage C Limit': 'Coverage C Limit',
        'Coverage D Limit': 'Coverage D Limit',
        'Seasonal Residence': 'Seasonal Residence',
        'Secondary Residence': 'Secondary Residence',
        'Months Unoccupied (Secondary or Seasonal Dwelling': 'Months Unoccupied',
        'Wood, Coal, or Pellet Stove': 'Wood, Coal, or Pellet Stove',
        'Log Home': 'Log Home',
        'Roof Covering': 'Roof Covering',
        'Retrofitted Dwelling Factor': 'Retrofitted Dwelling',
        'Loss History': 'Loss History',
        'Loss Free Discount': 'Loss Free Discount',
        'Home Buyer Discount': 'Home Buyer Discount',
        'Age of Insured': 'Age of Insured',
        'Coverage E and Coverage F Limits': 'Coverage E and F Limits',
        'Difference in Conditions Endorsement': 'DIC Endorsement',
        'Miles to Fire Department': 'Miles to Fire Department',
        'Roof Attached Structures': 'Roof Attached Structures',
        'Multi-Policy Discount': 'Multi-Policy',
        'Presence of Fence': 'Presence of Fence',
        'Presence of Pool': 'Presence of Pool',
        'Limited Water Damage Coverage': 'Limited Water',
        'Extended Dwelling Coverage': 'Extended Dwelling',
        'Stand Protect Discount': 'WF Mitigation',
        'Agreed Value': 'AgreedValue',
        'Firewise Community': 'Firewise',
        'Defensible space': 'Defensible Space',
        'Window Glass Type': 'Window Glass Type',
        'Rented to Others Surcharge': 'Rented to Other Surcharge',
        'Course of Construction Surcharge': 'Course of Construction',
        'Initial Fire Impact Probability': 'Initial Fire Impact Probability',
        'Resilient Home Discount': 'Resilient Home Discount',
        'Wildfire Score': 'Wildfire Score',
        'Individual Peril Premium': 'Total Premium Before Expenses',
        'Tier Factor1': 'Tier Factor' # Correcting based on observed spreadsheet data
    }

    # Apply the renaming to the index
    sheets_df = sheets_df.rename(index=factor_name_mapping)

    # Drop the row with 'Deductible Value' from the DataFrame if it exists
    if 'Deductible Value' in sheets_df.index:
        sheets_df = sheets_df.drop('Deductible Value')

    print(f"Successfully loaded and cleaned data from '{spreadsheet_name}' (range B10:P57) into a pandas DataFrame.")
    display(sheets_df.head())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet '{spreadsheet_name}' not found. Please check the name.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded and cleaned data from 'HO CA Rating Algorithm and Rate Tables(Main)' (range B10:P57) into a pandas DataFrame.


,P1,P2,P2-A,P2-B,P3,P4,P5,P6,P7,P8,P9,P10
Factor,,,,,,,,,,,,
Base Rate,3070.000,12947.000,14532.000,1438.000,1293.000,1677.000,621.000,12829.000,1106.000,1732.000,1612.000,419.000
Territory,0.999,1.451,1.000,1.451,0.494,0.507,0.766,0.398,0.405,1.389,0.227,0.682
Wildfire Score,1.000,2.571,1.000,2.571,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
ARF,1.000,1.000,3.136,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000
Tier Factor,0.595,1.000,1.000,1.000,0.753,0.661,0.891,0.865,0.818,0.971,0.696,1.000


In [ ]:
def compare_rate_tables(df1, df2, tolerance=1e-6):
    """
    Compares two pandas DataFrames for equality, considering potential float inaccuracies.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.
        tolerance (float): The absolute tolerance for comparing float values.

    Returns:
        bool: True if the DataFrames are equal, False otherwise.
        list: A list of messages detailing any differences found.
    """
    diff_messages = []

    # 1. Check if shapes are the same
    if df1.shape != df2.shape:
        diff_messages.append(f"Shapes are different: {df1.shape} vs {df2.shape}")
        # If shapes are different, further comparison might be misleading, so return early
        return False, diff_messages

    # 2. Check if indices are the same
    if not df1.index.equals(df2.index):
        diff_messages.append("Indices are different.")
        # Find differing index labels
        missing_in_df2 = df1.index.difference(df2.index)
        if not missing_in_df2.empty:
            diff_messages.append(f"  Labels in df1 not in df2: {list(missing_in_df2)}")
        missing_in_df1 = df2.index.difference(df1.index)
        if not missing_in_df1.empty:
            diff_messages.append(f"  Labels in df2 not in df1: {list(missing_in_df1)}")

    # 3. Check if columns are the same
    if not df1.columns.equals(df2.columns):
        diff_messages.append("Columns are different.")
        missing_in_df2 = df1.columns.difference(df2.columns)
        if not missing_in_df2.empty:
            diff_messages.append(f"  Columns in df1 not in df2: {list(missing_in_df2)}")
        missing_in_df1 = df2.columns.difference(df1.columns)
        if not missing_in_df1.empty:
            diff_messages.append(f"  Columns in df2 not in df1: {list(missing_in_df1)}")

    # If indices or columns are different, align them before comparing values
    common_index = df1.index.intersection(df2.index)
    common_columns = df1.columns.intersection(df2.columns)

    df1_common = df1.loc[common_index, common_columns]
    df2_common = df2.loc[common_index, common_columns]

    # 4. Compare values, handling NaNs and floats
    # Use .compare() for detailed differences in common elements
    comparison_result = df1_common.compare(df2_common)

    # If comparison_result is not empty, there are differences
    if not comparison_result.empty:
        diff_messages.append("Differences found in values:")
        # Filter for actual numerical differences beyond tolerance
        numerical_diffs_exist = False
        for idx in comparison_result.index:
            row_diffs = []
            for col in comparison_result.columns.levels[0]:
                val1 = comparison_result.loc[idx, (col, 'self')]
                val2 = comparison_result.loc[idx, (col, 'other')]

                # Check for NaNs equality (NaN is only equal to NaN here)
                if pd.isna(val1) and pd.isna(val2):
                    continue # Both are NaN, consider them equal
                if pd.isna(val1) or pd.isna(val2):
                    row_diffs.append(f"  {col}: {val1} vs {val2} (NaN difference)")
                    numerical_diffs_exist = True
                # Compare non-NaN numeric values with tolerance
                elif pd.api.types.is_numeric_dtype(type(val1)) and pd.api.types.is_numeric_dtype(type(val2)):
                    if not np.isclose(val1, val2, atol=tolerance, equal_nan=True):
                        row_diffs.append(f"  {col}: {val1} vs {val2} (numerical difference)")
                        numerical_diffs_exist = True
                # Compare non-numeric types directly
                elif val1 != val2:
                    row_diffs.append(f"  {col}: {val1} vs {val2} (type/value difference)")
                    numerical_diffs_exist = True

            if row_diffs:
                diff_messages.append(f"  Factor '{idx}':")
                diff_messages.extend(row_diffs)

        if not numerical_diffs_exist and not diff_messages:
            return True, ["DataFrames are equal within the given tolerance."]
        elif not numerical_diffs_exist and diff_messages:
             # Only index/column differences, but values are numerically close where comparable
            return False, diff_messages
    else:
        # If .compare() is empty, check if shapes, indices, and columns are all truly identical
        if df1.equals(df2): # Strict equality for structure and exact values
            return True, ["DataFrames are identical."]
        elif df1.index.equals(df2.index) and df1.columns.equals(df2.columns) and df1_common.shape == df1.shape: # Means only values could differ, and .compare caught nothing
            # This path is tricky, implies numerical close but maybe not identical for np.isclose if not used carefully
            return True, ["DataFrames are equal within the given tolerance."]

    if diff_messages:
        return False, diff_messages
    else:
        return True, ["DataFrames are identical."]

In [ ]:
# Assuming sheets_df and automated_rt are already defined and loaded
# If automated_rt is generated from your pretty_print_rate_table function, ensure it's run first

# Call the comparison function
are_equal, differences = compare_rate_tables(sheets_df, automated_rt)

if are_equal:
    print("The 'sheets_df' and 'automated_rt' DataFrames are considered equal.")
else:
    print("The 'sheets_df' and 'automated_rt' DataFrames are NOT equal. Differences found:")
    for diff in differences:
        print(f"- {diff}")

The 'sheets_df' and 'automated_rt' DataFrames are NOT equal. Differences found:
- Differences found in values:
-   Factor 'Age of Home':
-   P1: 1.018 vs 1.045 (numerical difference)
-   P2: 0.987 vs 0.998 (numerical difference)
-   P2-B: 0.987 vs 0.998 (numerical difference)
-   P3: 1.064 vs 1.071 (numerical difference)
-   P4: 1.002 vs 1.011 (numerical difference)
-   P5: 1.253 vs 1.224 (numerical difference)
-   P7: 1.064 vs 1.079 (numerical difference)
-   P8: 1.253 vs 1.224 (numerical difference)
-   P9: 1.332 vs 1.322 (numerical difference)
-   P10: 1.04 vs 1.045 (numerical difference)
-   Factor 'Total Premium Before Expenses':
-   P1: 871.0 vs 894.0 (numerical difference)
-   P2: 48950.0 vs 49495.0 (numerical difference)
-   P2-B: 5437.0 vs 5497.0 (numerical difference)
-   P3: 1068.0 vs 1075.0 (numerical difference)
-   P4: 72.0 vs 73.0 (numerical difference)
-   P5: 253.0 vs 247.0 (numerical difference)
-   P7: 116.0 vs 117.0 (numerical difference)
-   P8: 1217.0 vs 1189.0 (n

## Packages to Install
you should not have to run this again but just in case

In [ ]:
get_ipython().system('pip install gspread gspread-dataframe openpyxl')
print("Required libraries installed successfully.")